# 03 — Preditores do ENDURECIMENTO: Regressão (Cap. 6.3)Quais variáveis (regime, período, país, suporte) predizem o índice de purificação?**Modelo:** OLS + Regressão logística ordinal

In [ ]:
import pandas as pd, numpy as npimport statsmodels.api as smfrom scipy.stats import spearmanrimport matplotlib.pyplot as plt, seaborn as snssns.set_theme(style="whitegrid", font_scale=1.1)df = pd.read_csv('../data/processed/corpus_dataset.csv')INDICATORS = ['desincorporacao','rigidez_postural','dessexualizacao','uniformizacao_facial','heraldizacao','enquadramento_arquitetonico','apagamento_narrativo','monocromatizacao','serialidade','inscricao_estatal']print("="*60); print("  OLS: ENDURECIMENTO ~ regime"); print("="*60)X = pd.get_dummies(df['regime_iconocratico'], drop_first=True, dtype=float)X = sm.add_constant(X); y = df['purificacao_composto']model = sm.OLS(y, X).fit()print(model.summary2().tables[1].to_string())print(f"\nR² = {model.rsquared:.3f}, Adj. R² = {model.rsquared_adj:.3f}")

## 3.2 OLS com regime + suporte

In [ ]:
print("="*60); print("  OLS: ENDURECIMENTO ~ regime + medium"); print("="*60)predictors = pd.get_dummies(df['regime_iconocratico'], drop_first=True, dtype=float)med_dummies = pd.get_dummies(df['medium_norm'], prefix='med', drop_first=True, dtype=float)med_cols = [c for c in med_dummies.columns if med_dummies[c].sum() >= 5]predictors = pd.concat([predictors, med_dummies[med_cols]], axis=1)X_full = sm.add_constant(predictors)model_full = sm.OLS(df['purificacao_composto'], X_full).fit()print(model_full.summary2().tables[1].to_string())print(f"\nR² = {model_full.rsquared:.3f}, Adj. R² = {model_full.rsquared_adj:.3f}")

## 3.3 Dessexualização como preditor — teste do threshold

In [ ]:
print("="*60); print("  Correlação de cada indicador com o composto"); print("="*60)for ind in INDICATORS:    rho, p = spearmanr(df[ind], df['purificacao_composto'])    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'    print(f"  {ind:<30s} ρ = {rho:.3f}  p = {p:.4f} {sig}")fig, ax = plt.subplots(figsize=(8, 5))palette = {'fundacional': '#e74c3c', 'normativo': '#3498db', 'militar': '#2c3e50'}for regime in ['fundacional', 'normativo', 'militar']:    sub = df[df.regime_iconocratico == regime]    ax.scatter(sub['dessexualizacao'] + np.random.normal(0, 0.08, len(sub)),              sub['purificacao_composto'], alpha=0.6, label=regime.capitalize(), color=palette[regime], s=50)ax.set_xlabel('Dessexualização (0-3)'); ax.set_ylabel('Score composto ENDURECIMENTO')ax.set_title('Dessexualização vs. ENDURECIMENTO por regime'); ax.legend()plt.tight_layout(); plt.savefig('../data/processed/fig_07_dessex_vs_endurec.png', dpi=150, bbox_inches='tight'); plt.show()

## 3.4 Importância relativa dos indicadores

In [ ]:
# Which indicators contribute most to variance?from scipy.stats import spearmanrrhos = [(INDICATOR_LABELS[ind] if ind in INDICATOR_LABELS else ind, spearmanr(df[ind], df['purificacao_composto'])[0]) for ind in INDICATORS]rhos.sort(key=lambda x: x[1], reverse=True)fig, ax = plt.subplots(figsize=(10, 5))ax.barh([r[0] for r in rhos], [r[1] for r in rhos], color='steelblue')ax.set_xlabel('Spearman ρ com ENDURECIMENTO composto'); ax.set_title('Contribuição de cada indicador ao ENDURECIMENTO')ax.axvline(0, color='grey', ls='--')plt.tight_layout(); plt.savefig('../data/processed/fig_08_indicator_importance.png', dpi=150, bbox_inches='tight'); plt.show()